In [0]:
dbutils.widgets.text("catalog", "shopstream_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.daily_revenue AS
SELECT DATE(order_ts) AS order_date,
COUNT(DISTINCT order_id) AS orders,
SUM(quantity) AS units_sold,
ROUND(SUM(line_revenue), 2) AS revenue
FROM {catalog}.silver.orders
WHERE status = 'completed'
GROUP BY DATE(order_ts)
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.category_performance AS
SELECT p.category,
COUNT(DISTINCT o.order_id) AS orders,
SUM(o.quantity) AS units_sold,
ROUND(SUM(o.line_revenue), 2) AS revenue,
ROUND(SUM(o.quantity * p.unit_margin), 2) AS gross_margin
FROM {catalog}.silver.orders o
JOIN {catalog}.silver.products p USING (product_id)
WHERE o.status = 'completed'
GROUP BY p.category
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.customer_ltv AS
SELECT c.customer_id, c.name, c.country, c.signup_channel,
COUNT(DISTINCT o.order_id) AS lifetime_orders,
ROUND(SUM(o.line_revenue), 2) AS lifetime_revenue
FROM {catalog}.silver.orders o
JOIN {catalog}.silver.customers c
ON o.customer_id = c.customer_id AND c.is_current = true
WHERE o.status = 'completed'
GROUP BY c.customer_id, c.name, c.country, c.signup_channel
""")

In [0]:
%sql
SELECT * FROM
shopstream_dev.gold.category_performance ORDER BY revenue DESC